In [16]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA H100 NVL


In [3]:
# ===== INSTALL REQUIRED PACKAGES IN COLAB =====

import sys
import subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# core packages for your TFT + BaseModel pipeline
pip_install(
    "lightning",
    "pytorch-forecasting",
    "pandas",
    "numpy",
    "matplotlib"
)

print("All required packages installed.")


All required packages installed.


In [17]:
import os
import random
import pickle
import json
import torch

from abc import ABC, abstractmethod
import numpy as np
import pandas as pd

class BaseModel(ABC):

    QUANTILES   = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]
    PRED_LENGTH = 28
    SEED        = 25
    TARGET_START = 1914

    def __init__(self, data_dir="data", output_dir="outputs"):
        random.seed(self.SEED)
        np.random.seed(self.SEED)
        torch.manual_seed(self.SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.data_dir   = data_dir
        self.output_dir = output_dir
        os.makedirs(data_dir,   exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        self.train_raw = self.val_raw = self.test_raw = None
        self.calendar  = self.prices  = self.item_weights = None
        self.train_processed = self.val_processed = self.test_processed = None
        self.model = None

    @property
    @abstractmethod
    def model_name(self) -> str: ...

    @abstractmethod
    def preprocess(self): ...
    # Input:  self.train_raw, self.val_raw, self.test_raw
    # Output: self.train_processed, self.val_processed, self.test_processed
    #         (format is model-specific: tensors, lgb.Dataset, DataFrames, etc.)

    @abstractmethod
    def train(self): ...
    # Input:  self.train_processed, self.val_processed
    # Must:
    #   1. Set self.model
    #   2. Save weights → output_dir/{model_name}.pth (or .pkl for LGBM)

    @abstractmethod
    def predict(self): ...
    # Input: self.test_processed, self.output_dir
    # Must:
    # 1. Load model from: output_dir/{model_name}.pth/pkl
    # 2. Only use d_1886-d_1913 as context (if needed)
    # 3. Predict 9 quantiles for d_1914-d_1941 in preds_df: a DataFrame (30490 * 28 rows) with columns:
    #     id | day_ahead (1-28) | q0.025 | q0.05 | q0.1 | q0.25 | q0.5 | q0.75 | q0.9 | q0.95 | q0.975
    # 4. Sort predictions by id, day_ahead
    # 5. Make sure quantiles are non-decreasing and >= 0
    # 6. Save predictions → output_dir/{model_name}_predictions.csv
    # Output: preds_df

    # Shared methods

    def load_and_split_data(self):
        """
        Downloads (or loads from cache) M5 data and processes it into long-format dataframes (1 row per item and day).
        Split into train (d_1-d_1773), val (d_1774-d_1885), and test (d_1886-d_1941).
        Save raw splits to cache for loading when training models.

        Output: self.train_raw, self.val_raw, self.test_raw, self.item_weights
        """
        # Step 1: load data
        cache = os.path.join(self.data_dir, "raw_split.pkl")

        if os.path.exists(cache):
            with open(cache, "rb") as f:
                d = pickle.load(f)
            print("Loaded cached data splits.")

        else:
            base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
            sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
            calendar = pd.read_csv(f"{base}/calendar.csv")
            prices   = pd.read_csv(f"{base}/sell_prices.csv")
            print("Downloaded M5 data.")

            # Step 2: melt sales to long format
            id_cols  = [c for c in sales.columns if not c.startswith("d_")]
            day_cols = [c for c in sales.columns if c.startswith("d_")]
            sales_long = sales[id_cols + day_cols].melt(
                id_vars=id_cols, var_name='d', value_name='sales'
            )
            sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

            # Step 3: merge calendar data and set dtypes
            for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
                calendar[col] = calendar[col].fillna('none').astype('category')
            calendar['weekday'] = calendar['weekday'].astype('category')
            calendar['date']    = pd.to_datetime(calendar['date'])
            for col in ['snap_CA', 'snap_TX', 'snap_WI', 'wday', 'month']:
                calendar[col] = calendar[col].astype('int8')
            calendar['year'] = calendar['year'].astype('int16')  # fix year encoding to handle years 2011-2016
            sales_long = sales_long.merge(calendar, on='d', how='left')

            # Step 4: merge daily prices
            sales_long = sales_long.merge(
                prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
                on=['store_id', 'item_id', 'wm_yr_wk'], how='left'
            )

            # Step 5: add is_available flag and forward-fill missing prices
            sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')
            sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)
            sales_long['sell_price'] = (
                sales_long.groupby('id')['sell_price']
                .transform(lambda x: x.ffill().fillna(0.0))
            )

            # Step 6: check missing values
            missing = sales_long.isnull().sum()
            missing = missing[missing > 0]
            assert len(missing) == 0, f"Missing values found:\n{missing}"

            # Step 7: split into train/val/test
            total_days = len(day_cols)
            train_end  = total_days - 6 * self.PRED_LENGTH   # 1773
            val_end    = total_days - 2 * self.PRED_LENGTH   # 1885

            train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
            val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
            test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

            assert train_raw['d_num'].nunique() == 1773
            assert val_raw['d_num'].nunique()   == 112
            assert test_raw['d_num'].nunique()  == 56
            print(f"Train: d_1-d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

            # Step 8: calculate revenue weights on last 28 training days only
            last28 = train_raw[train_raw['d_num'] > train_end - 28]
            train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
            item_weights = (train_rev / train_rev.sum()).rename('weight')

            # Step 9: save data splits to cache
            d = dict(train_raw=train_raw, val_raw=val_raw, test_raw=test_raw,
                        item_weights=item_weights)

            # save all splits and weights into raw_split.pkl
            with open(cache, "wb") as f:
                pickle.dump(d, f)
            print(f"Cached to {cache}")

        self.train_raw    = d["train_raw"]
        self.val_raw      = d["val_raw"]
        self.test_raw     = d["test_raw"]
        self.item_weights = d["item_weights"]

        return self.train_raw, self.val_raw, self.test_raw, self.item_weights

    # Evaluation

    def _build_pinball_tensor(self, preds_df: pd.DataFrame):
        """
        Shared setup for WSPL and CRPS. Requires self.train_raw, self.test_raw.
        Returns y_mat (N,28), q_arr (N,9,28), ids, scale.
        """
        q_cols  = [f"q{q}" for q in self.QUANTILES]

        # Extract test targets for d_1914-d_1941 (28 days)
        test_targets = (
            self.test_raw[self.test_raw['d_num'] >= self.TARGET_START]
            .pivot(index='id', columns='d_num', values='sales')
            .sort_index()
        )
        ids = test_targets.index.tolist()
        y_mat = test_targets.values.astype(np.float32)  # Shape: (N_series, 28)

        # Reshape predictions to match
        preds_pivot = (
            preds_df.set_index(['id', 'day_ahead'])[q_cols]
            .unstack('day_ahead')
            .sort_index(axis=1)
            .loc[ids]
)
        q_arr = preds_pivot.values.reshape(len(ids), len(self.QUANTILES), self.PRED_LENGTH)

        # Scale from train
        train_s = self.train_raw.sort_values(['id', 'd_num']).copy()
        train_s['prev'] = train_s.groupby('id')['sales'].shift(1)
        scale = (
            train_s.dropna(subset=['prev'])
            .assign(abs_diff=lambda df: (df['sales'] - df['prev']).abs())
            .groupby('id')['abs_diff']
            .mean()
            .reindex(ids)  # Align scale to test data IDs
            .clip(lower=1e-8)
        )

        # Checks for debugging
        assert y_mat.shape == (len(ids), 28), f"y_mat shape mismatch: {y_mat.shape}"
        assert q_arr.shape == (len(ids), 9, 28), f"q_arr shape mismatch: {q_arr.shape}"

        return y_mat, q_arr, ids, scale

    def compute_wspl(self, y_mat, q_arr, ids, scale) -> float:
        """Weighted scaled pinball loss across 9 quantiles and 28 forecast days."""
        q_vals  = np.array(self.QUANTILES)
        errors  = y_mat[:, None, :] - q_arr
        pinball = np.maximum(q_vals[None, :, None] * errors,
                             (q_vals[None, :, None] - 1) * errors)
        wspl_per_series = pinball.mean(axis=(1, 2)) / scale.reindex(ids).values
        weights         = self.item_weights.reindex(ids).fillna(0).values
        return float(np.dot(weights, wspl_per_series))

    def compute_crps(self, y_mat, q_arr, ids) -> float:
        """Weighted CRPS approximated as 2x mean pinball loss across quantiles and forecast days."""
        q_vals  = np.array(self.QUANTILES)
        errors  = y_mat[:, None, :] - q_arr
        pinball = np.maximum(q_vals[None, :, None] * errors,
                            (q_vals[None, :, None] - 1) * errors)
        crps_per_series = 2 * pinball.mean(axis=(1, 2))
        weights = self.item_weights.reindex(ids).fillna(0).values
        return float(np.dot(weights, crps_per_series))

    def compute_coverage(self, preds_df: pd.DataFrame, y_mat, ids) -> dict:
        """
        Coverage error (actual - nominal) for 4 prediction intervals.
        Positive = over-coverage, negative = under-coverage.
        """
        intervals = {
            0.50: ("q0.25",  "q0.75"),
            0.80: ("q0.1",   "q0.9"),
            0.90: ("q0.05",  "q0.95"),
            0.95: ("q0.025", "q0.975"),
        }
        preds_indexed = (
            preds_df.set_index(['id', 'day_ahead'])
            .reindex(pd.MultiIndex.from_product(
                [ids, range(1, self.PRED_LENGTH + 1)],
                names=['id', 'day_ahead']))
        )
        coverage_errors = {}
        for nominal, (lower_col, upper_col) in intervals.items():
            lower   = preds_indexed[lower_col].values.reshape(len(ids), self.PRED_LENGTH)
            upper   = preds_indexed[upper_col].values.reshape(len(ids), self.PRED_LENGTH)
            covered = ((y_mat >= lower) & (y_mat <= upper)).mean()
            coverage_errors[f"coverage_error_{int(nominal*100)}pct"] = round(
                float(covered - nominal), 6)
        return coverage_errors

    def _validate_preds(self, preds_df):
        """
        Validate predictions for easier debugging of inference pipeline.
        """
        expected_ids = self.test_raw['id'].unique()
        q_cols = [f"q{q}" for q in self.QUANTILES]
        assert set(expected_ids).issubset(set(preds_df['id'])), "Missing IDs"
        assert set(preds_df['day_ahead'].unique()) == set(range(1, 29)), "day_ahead must be 1-28"
        assert all(c in preds_df.columns for c in q_cols), "Missing quantile columns"
        assert (preds_df[q_cols].diff(axis=1).iloc[:, 1:] >= 0).all().all(), "Quantiles non-monotonic"
        assert (preds_df[q_cols] >= 0).all().all(), "Negative quantile predictions"

    def evaluate(self, preds_df: pd.DataFrame) -> dict:
        """
        Compute WSPL, CRPS and coverage error from a predictions DataFrame.
        Saves results to output_dir/{model_name}_results.json.
        Requires: raw_split.pkl in data_dir. Run load_and_split_data() first.
        """
        self._validate_preds(preds_df)
        # Step 1: load data and weights
        cache = os.path.join(self.data_dir, "raw_split.pkl")
        assert os.path.exists(cache), "Run load_and_split_data() first."
        with open(cache, "rb") as f:
            d = pickle.load(f)
        self.train_raw    = d["train_raw"]
        self.test_raw     = d["test_raw"]
        self.item_weights = d["item_weights"]

        # Step 2: build pinball tensor
        y_mat, q_arr, ids, scale = self._build_pinball_tensor(preds_df)

        # Step 3: compute evaluation metrics
        results = {
            "model" : self.model_name,
            "wspl"  : round(self.compute_wspl(y_mat, q_arr, ids, scale), 6),
            "crps"  : round(self.compute_crps(y_mat, q_arr, ids), 6),
            **self.compute_coverage(preds_df, y_mat, ids),
        }

        # Step 4: save results to json
        out_path = os.path.join(self.output_dir, f"{self.model_name}_results.json")
        with open(out_path, "w") as f:
            json.dump(results, f, indent=2)
        return results

    # Pipelines

    def run_training_pipeline(self):
    # Combines loading and splitting data, preprocessing, and training into a single pipeline
        self.load_and_split_data()
        print("Finished shared data processing.")
        self.preprocess()
        print("Finished model-specific data processing.")
        self.train()
        print("Finished model training.")

    def run_inference_pipeline(self):
    # Combines inference and evaluation into a single pipeline
        self.load_and_split_data()  # needs train_raw and test_raw, if training pipeline was not run before this
        preds_df = self.predict()
        print("Finished model inference.")
        results = self.evaluate(preds_df)
        print("Finished model evaluation.")
        return results

In [ ]:
import os
import json
import pickle
import warnings
from typing import Dict, Any

import numpy as np
import pandas as pd
import torch

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss


# THIS CODE INCLUDES THE TFT SUBCLASS!!!

warnings.filterwarnings("ignore")


class TFTModel(BaseModel):
    """
    Temporal Fusion Transformer model compatible with BaseModel.

    Key design:
    - Uses BaseModel.load_and_split_data() for shared M5 processing.
    - Builds TFT datasets from train/val/test raw splits.
    - Trains on d_1-d_1773.
    - Validates on the final 28 days of the val block:
        encoder context: d_1830-d_1857
        decoder target:  d_1858-d_1885
    - Predicts on the final 28 days of the test block:
        encoder context: d_1886-d_1913
        decoder target:  d_1914-d_1941
    """

    MAX_ENCODER_LENGTH = 28
    MAX_PREDICTION_LENGTH = 28

    def __init__(
        self,
        data_dir: str = "data",
        output_dir: str = "outputs",
        batch_size: int = 128,
        num_workers: int = 2,
        max_epochs: int = 10,
        learning_rate: float = 1e-3,
        hidden_size: int = 32,
        attention_head_size: int = 4,
        dropout: float = 0.1,
        hidden_continuous_size: int = 16,
        gradient_clip_val: float = 0.1,
    ):
        super().__init__(data_dir=data_dir, output_dir=output_dir)

        self.batch_size = batch_size
        self.num_workers = num_workers
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.attention_head_size = attention_head_size
        self.dropout = dropout
        self.hidden_continuous_size = hidden_continuous_size
        self.gradient_clip_val = gradient_clip_val

        self.dataset_objects: Dict[str, Any] = {}
        self.dataloaders: Dict[str, Any] = {}

    @property
    def model_name(self) -> str:
        return "tft"

    def _checkpoint_path(self) -> str:
        return os.path.join(self.output_dir, f"{self.model_name}.ckpt")

    def _config_path(self) -> str:
        return os.path.join(self.output_dir, f"{self.model_name}_config.json")

    def _preprocess_cache_path(self) -> str:
        return os.path.join(self.output_dir, f"{self.model_name}_preprocess.pkl")

    def _metrics_path(self) -> str:
        return os.path.join(self.output_dir, f"{self.model_name}_train_metrics.json")

    def _prepare_full_dataframe(self) -> pd.DataFrame:
        """
        Combine train/val/test raw splits into one dataframe for consistent TFT encoding.
        """
        full_df = pd.concat(
            [self.train_raw, self.val_raw, self.test_raw],
            axis=0,
            ignore_index=True
        ).copy()

        full_df["series_id"] = full_df["id"].astype(str)
        full_df["time_idx"] = full_df["d_num"].astype(int)
        full_df["sales"] = full_df["sales"].astype(float)
        full_df["sell_price"] = full_df["sell_price"].astype(float)
        full_df["is_available"] = full_df["is_available"].astype(float)

        # Keep raw string categoricals for TFT
        categorical_cols = [
            "series_id",
            "item_id", "dept_id", "cat_id", "store_id", "state_id",
            "weekday", "event_name_1", "event_type_1", "event_name_2", "event_type_2"
        ]
        for col in categorical_cols:
            full_df[col] = full_df[col].astype(str)

        # # Known reals
        # known_real_cols = [
        #     "time_idx", "sell_price", "is_available",
        #     "snap_CA", "snap_TX", "snap_WI",
        #     "wday", "month", "year"
        # ]
        # for col in known_real_cols:
        #     full_df[col] = full_df[col].astype(float)

        # time_idx must stay integer for TimeSeriesDataSet
        full_df["time_idx"] = full_df["time_idx"].astype("int64")

        # known real-valued inputs
        known_real_cols = [
    "sell_price", "is_available",
    "snap_CA", "snap_TX", "snap_WI",
    "wday", "month", "year"
]
        for col in known_real_cols:
            full_df[col] = full_df[col].astype(float)



        
        full_df = full_df.sort_values(["series_id", "time_idx"]).reset_index(drop=True)
        return full_df

    def preprocess(self):
        """
        Build TFT TimeSeriesDataSet objects and dataloaders.

        We create:
        - training dataset using only train period (<= 1773)
        - validation prediction dataset for decoder horizon d_1858-d_1885
        - test prediction dataset for decoder horizon d_1914-d_1941
        """
        full_df = self._prepare_full_dataframe()

        train_end = int(self.train_raw["d_num"].max())      # 1773
        val_start = int(self.val_raw["d_num"].min())        # 1774
        val_end = int(self.val_raw["d_num"].max())          # 1885
        test_start = int(self.test_raw["d_num"].min())      # 1886
        test_end = int(self.test_raw["d_num"].max())        # 1941

        # Final validation target horizon inside val split:
        # encoder: 1830-1857, decoder: 1858-1885
        val_prediction_start = val_end - self.PRED_LENGTH + 1  # 1858

        # Final test target horizon:
        # encoder: 1886-1913, decoder: 1914-1941
        test_prediction_start = self.TARGET_START  # 1914

        training_cut_df = full_df[full_df["time_idx"] <= train_end].copy()

        training = TimeSeriesDataSet(
            training_cut_df,
            time_idx="time_idx",
            target="sales",
            group_ids=["series_id"],

            min_encoder_length=self.MAX_ENCODER_LENGTH,
            max_encoder_length=self.MAX_ENCODER_LENGTH,
            min_prediction_length=self.MAX_PREDICTION_LENGTH,
            max_prediction_length=self.MAX_PREDICTION_LENGTH,

            static_categoricals=[
                "item_id", "dept_id", "cat_id", "store_id", "state_id"
            ],

            time_varying_known_categoricals=[
                "weekday", "event_name_1", "event_type_1", "event_name_2", "event_type_2"
            ],

            time_varying_known_reals=[
                "time_idx", "sell_price", "is_available",
                "snap_CA", "snap_TX", "snap_WI",
                "wday", "month", "year"
            ],

            time_varying_unknown_reals=[
                "sales"
            ],

            target_normalizer=GroupNormalizer(
                groups=["series_id"],
                transformation="softplus"
            ),

            add_relative_time_idx=True,
            add_target_scales=True,
            add_encoder_length=True,
        )

        # Validation dataframe must include enough context before 1858
        val_context_start = val_prediction_start - self.MAX_ENCODER_LENGTH  # 1830
        val_df = full_df[
            (full_df["time_idx"] >= val_context_start) &
            (full_df["time_idx"] <= val_end)
        ].copy()

        validation = TimeSeriesDataSet.from_dataset(
            training,
            val_df,
            predict=True,
            stop_randomization=True
        )

        # Test dataframe must include enough context before 1914
        test_context_start = test_prediction_start - self.MAX_ENCODER_LENGTH  # 1886
        test_df = full_df[
            (full_df["time_idx"] >= test_context_start) &
            (full_df["time_idx"] <= test_end)
        ].copy()

        test_dataset = TimeSeriesDataSet.from_dataset(
            training,
            test_df,
            predict=True,
            stop_randomization=True
        )

        train_dataloader = training.to_dataloader(
            train=True,
            batch_size=self.batch_size,
            num_workers=self.num_workers
        )

        val_dataloader = validation.to_dataloader(
            train=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers
        )

        test_dataloader = test_dataset.to_dataloader(
            train=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers
        )

        self.train_processed = training
        self.val_processed = validation
        self.test_processed = test_dataset

        self.dataset_objects = {
            "training": training,
            "validation": validation,
            "test": test_dataset,
            "full_df": full_df,
            "val_prediction_start": val_prediction_start,
            "test_prediction_start": test_prediction_start,
        }

        self.dataloaders = {
            "train": train_dataloader,
            "val": val_dataloader,
            "test": test_dataloader,
        }

        preprocess_info = {
            "max_encoder_length": self.MAX_ENCODER_LENGTH,
            "max_prediction_length": self.MAX_PREDICTION_LENGTH,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end,
            "val_prediction_start": val_prediction_start,
            "test_prediction_start": test_prediction_start,
        }

        with open(self._preprocess_cache_path(), "wb") as f:
            pickle.dump(preprocess_info, f)

        with open(self._config_path(), "w") as f:
            json.dump(
                {
                    "model_name": self.model_name,
                    "quantiles": self.QUANTILES,
                    "batch_size": self.batch_size,
                    "num_workers": self.num_workers,
                    "max_epochs": self.max_epochs,
                    "learning_rate": self.learning_rate,
                    "hidden_size": self.hidden_size,
                    "attention_head_size": self.attention_head_size,
                    "dropout": self.dropout,
                    "hidden_continuous_size": self.hidden_continuous_size,
                    "max_encoder_length": self.MAX_ENCODER_LENGTH,
                    "max_prediction_length": self.MAX_PREDICTION_LENGTH,
                },
                f,
                indent=2
            )

        print(f"Training samples:   {len(training)}")
        print(f"Validation samples: {len(validation)}")
        print(f"Test samples:       {len(test_dataset)}")

    def train(self):
        """
        Train TFT and save best checkpoint to outputs/tft.ckpt
        """
        assert self.train_processed is not None, "Run preprocess() first."
        assert self.val_processed is not None, "Run preprocess() first."

        training = self.train_processed
        train_dataloader = self.dataloaders["train"]
        val_dataloader = self.dataloaders["val"]

        tft = TemporalFusionTransformer.from_dataset(
            training,
            learning_rate=self.learning_rate,
            hidden_size=self.hidden_size,
            attention_head_size=self.attention_head_size,
            dropout=self.dropout,
            hidden_continuous_size=self.hidden_continuous_size,
            output_size=len(self.QUANTILES),
            loss=QuantileLoss(quantiles=self.QUANTILES),
            log_interval=10,
            reduce_on_plateau_patience=3,
        )

        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            min_delta=1e-4,
            patience=3,
            verbose=True,
            mode="min"
        )

        lr_logger = LearningRateMonitor()

        checkpoint_callback = ModelCheckpoint(
            dirpath=self.output_dir,
            filename=f"{self.model_name}-best-temp",
            monitor="val_loss",
            mode="min",
            save_top_k=1
        )

        # trainer = pl.Trainer(
        #     max_epochs=self.max_epochs,
        #     accelerator="gpu" if torch.cuda.is_available() else "cpu",
        #     devices=1,
        #     gradient_clip_val=self.gradient_clip_val,
        #     callbacks=[lr_logger, early_stop_callback, checkpoint_callback],
        #     log_every_n_steps=10,
        # )
        trainer = pl.Trainer(
    max_epochs=self.max_epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=self.gradient_clip_val,
    callbacks=[lr_logger, early_stop_callback, checkpoint_callback],
    log_every_n_steps=10,
    limit_train_batches=2000,
    limit_val_batches=1.0,
)

        trainer.fit(
            tft,
            train_dataloaders=train_dataloader,
            val_dataloaders=val_dataloader
        )

        best_model_path = checkpoint_callback.best_model_path
        if not best_model_path:
            raise RuntimeError("No best checkpoint was saved during training.")

        best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)
        self.model = best_tft

        final_ckpt_path = self._checkpoint_path()
        if os.path.abspath(best_model_path) != os.path.abspath(final_ckpt_path):
            import shutil
            shutil.copy(best_model_path, final_ckpt_path)

        metrics = {
            "best_model_path": final_ckpt_path,
            "best_val_loss": float(checkpoint_callback.best_model_score.cpu().item())
            if checkpoint_callback.best_model_score is not None else None,
        }

        with open(self._metrics_path(), "w") as f:
            json.dump(metrics, f, indent=2)

        print(f"Best checkpoint saved to: {final_ckpt_path}")
        print(f"Best val_loss: {metrics['best_val_loss']}")

    def predict(self) -> pd.DataFrame:
        """
        Load saved TFT checkpoint and generate 9 quantile predictions for d_1914-d_1941.
        """
        checkpoint_path = self._checkpoint_path()
        assert os.path.exists(checkpoint_path), f"Checkpoint not found: {checkpoint_path}"

        # Rebuild processed datasets if this is a fresh inference run
        if self.test_processed is None or "test" not in self.dataloaders:
            self.preprocess()

        test_dataloader = self.dataloaders["test"]

        best_tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint_path)
        best_tft.eval()
        self.model = best_tft

        predictions = best_tft.predict(
            test_dataloader,
            mode="raw",
            return_x=True,
            return_index=True
        )

        raw_predictions = predictions.output
        x = predictions.x
        index = predictions.index

        pred_tensor = raw_predictions["prediction"].detach().cpu().numpy()
        actuals = x["decoder_target"].detach().cpu().numpy()

        rows = []
        batch_size, horizon, num_q = pred_tensor.shape

        expected_horizon = self.PRED_LENGTH
        expected_num_q = len(self.QUANTILES)

        assert horizon == expected_horizon, f"Expected horizon {expected_horizon}, got {horizon}"
        assert num_q == expected_num_q, f"Expected {expected_num_q} quantiles, got {num_q}"

        for i in range(batch_size):
            series_id = index.iloc[i]["series_id"]

            for t in range(horizon):
                row = {
                    "id": series_id,
                    "day_ahead": t + 1,
                }

                for j, q in enumerate(self.QUANTILES):
                    row[f"q{q}"] = float(pred_tensor[i, t, j])

                rows.append(row)

        preds_df = pd.DataFrame(rows)

        q_cols = [f"q{q}" for q in self.QUANTILES]

        preds_df[q_cols] = (
            preds_df[q_cols]
            .clip(lower=0)
            .apply(
                lambda row: np.maximum.accumulate(row.values),
                axis=1,
                result_type="expand"
            )
            .set_axis(q_cols, axis=1)
        )

        preds_df = (
            preds_df[["id", "day_ahead"] + q_cols]
            .sort_values(["id", "day_ahead"])
            .reset_index(drop=True)
        )

        out_path = os.path.join(self.output_dir, f"{self.model_name}_predictions.csv")
        preds_df.to_csv(out_path, index=False)
        print(f"Predictions saved to {out_path}")

        return preds_df

In [19]:
# ===== TEST CELL: QUICK COLAB SMOKE TEST =====

m = TFTModel(
    data_dir="data",
    output_dir="outputs",
    batch_size=64,
    num_workers=2,
    max_epochs=2,              # quick test
    learning_rate=1e-3,
    hidden_size=16,            # smaller = safer/faster
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=8,
    gradient_clip_val=0.1,
)

# train
m.run_training_pipeline()

# inference + evaluation
results = m.run_inference_pipeline()

print("\nFinal results:")
print(json.dumps(results, indent=2))

Loaded cached data splits.
Finished shared data processing.
Training samples:   52381820
Validation samples: 30490
Test samples:       30490
Finished model-specific data processing.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │ 49.3 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    224 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.5 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  9.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  8.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    676 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    153 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 82.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 82.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 580                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 0.511
Metric val_loss improved by 0.008 >= min_delta = 0.0001. New best score: 0.503
`Trainer.fit` stopped: `max_epochs=2` reached.


Best checkpoint saved to: outputs/tft.ckpt
Best val_loss: 0.5027132630348206
Finished model training.
Loaded cached data splits.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predictions saved to outputs/tft_predictions.csv
Finished model inference.
Finished model evaluation.

Final results:
{
  "model": "tft",
  "wspl": 0.561633,
  "crps": 1.231692,
  "coverage_error_50pct": -0.333464,
  "coverage_error_80pct": -0.513797,
  "coverage_error_90pct": -0.528431,
  "coverage_error_95pct": -0.543724
}
